## What will we learn from this project?
* Bivariate Analysis

* Comparison of Features
* Many visualization techniques
* Many of the popular Machine Learning techniques
* confusion matrix


### Import Library

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns 
import matplotlib.pyplot as plt


import graphviz
import itertools


from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
from sklearn import tree

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

<img src= "https://idsb.tmgrup.com.tr/ly/uploads/images/2021/10/07/150369.jpeg">

### EDA

In [ ]:
df = pd.read_csv('/kaggle/input/breast-cancer-wisconsin-data/data.csv')

In [ ]:
df.shape

In [ ]:
df.tail(10)

In [ ]:
df.groupby('diagnosis').size()

In [ ]:
df.isnull().sum()

## Preprocessing

* **As you can see we have two redundant features. We need to drop these columns 'id' and 'unnamed 32'**
* **We have two labels M : Malign and B : Benign**

In [ ]:
df = df.drop(["Unnamed: 32","id"],axis = 1)
X = df.drop("diagnosis",axis = 1)
Y = df['diagnosis']

X.head()

### Visualization

In [ ]:
M = df[df.diagnosis == "M"]
B = df[df.diagnosis == "B"]
# scatter plot
plt.scatter(M.radius_mean,M.texture_mean,color="red",label="Malign",alpha= 0.3)
plt.scatter(B.radius_mean,B.texture_mean,color="green",label="Benign",alpha= 0.3)
plt.xlabel("radius_mean")
plt.ylabel("texture_mean")
plt.legend()
plt.show()

In [ ]:
# Box plot
fig, axes =  plt.subplots(1, 2, figsize=(15,5))
sns.boxplot(ax = axes[0], x=df.diagnosis, y=df['area_mean'] ,palette="turbo")
axes[0].set_title('Size Difference')

sns.boxplot(ax = axes[1], x=df.diagnosis, y=df['perimeter_mean'] ,palette="PRGn")
axes[1].set_title('Size Difference')

plt.show()


In [ ]:
# let's optimize our data
x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size = 0.2,random_state = 42)

### Let's look at the correlation between the attributes

In [ ]:
fig =  plt.subplots(1, 1, figsize=(10,8))
grp = sns.heatmap(x_train.corr(),cmap="Spectral_r",annot = False)

### Creat Models

In [ ]:
model = tree.DecisionTreeClassifier(max_depth= 3 , min_samples_leaf=12)

model.fit(x_train,y_train)

ACC = model.score(x_train,y_train)    
print("Train dataset Accuracy = %"+ str(ACC*100))

ACC = model.score(x_test,y_test)    
print("Test dataset Accuracy = %"+ str(ACC*100))

y_pred = model.predict(x_test)

ACC = metrics.accuracy_score(y_test,y_pred)    
print("Accuracy = %"+ str(ACC*100))

In [ ]:
# Confusion Matrix
import scikitplot.metrics as splt

splt.plot_confusion_matrix(y_test,y_pred,figsize=(7,7))
plt.show()

In [ ]:
#

attributes_name = X.columns.values

def Decision_Tree_draw (mdl,names):
    ds = tree.export_graphviz(mdl,out_file = None,
                             feature_names = names,
                             class_names = ['Malign' ,'Benign'],
                             filled = False, rounded = True,
                             special_characters =False)
    
    grp = graphviz.Source(ds)
    return grp

Decision_Tree_draw(model,attributes_name)

### So how do we increase our accuracy rate?

**I think we can discard features with high correlations.**

In [ ]:
X = X.drop(['smoothness_worst','perimeter_worst','radius_worst'],axis =1)

x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size = 0.1,random_state = 42)

model = tree.DecisionTreeClassifier(max_depth= 3 , min_samples_leaf=12)

model.fit(x_train,y_train)

ACC = model.score(x_train,y_train)    
print("Train dataset Accuracy = %"+ str(ACC*100))

ACC = model.score(x_test,y_test)    
print("Test dataset Accuracy = %"+ str(ACC*100))


### Strongly correlated features cause poor generalization performance on the test set due to high variance and less model interpretability.

### Another method is to use other classification algorithms


In [ ]:
# RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

model.fit(x_train,y_train)

ACC = model.score(x_train,y_train)    
print("Train dataset Accuracy = %"+ str(ACC*100))

ACC = model.score(x_test,y_test)    
print("Test dataset Accuracy = %"+ str(ACC*100))

y_pred = model.predict(x_test)

ACC = metrics.accuracy_score(y_test,y_pred)    
print("Accuracy = %"+ str(ACC*100))


In [ ]:
# K-NN
from sklearn.neighbors import KNeighborsClassifier

model =KNeighborsClassifier(3)

model.fit(x_train,y_train)

ACC = model.score(x_train,y_train)    
print("Train dataset Accuracy = %"+ str(ACC*100))

ACC = model.score(x_test,y_test)    
print("Test dataset Accuracy = %"+ str(ACC*100))

y_pred = model.predict(x_test)

ACC = metrics.accuracy_score(y_test,y_pred)    
print("Accuracy = %"+ str(ACC*100))


In [ ]:
# support vector machine
from sklearn.svm import SVC

svm = SVC(random_state=1)

svm.fit(x_train,y_train)

ACC = svm.score(x_train,y_train)    
print("Train dataset Accuracy = %"+ str(ACC*100))

ACC = svm.score(x_test,y_test)    
print("Test dataset Accuracy = % "+ str(ACC*100))

In [ ]:
from sklearn.ensemble import BaggingClassifier

clf = BaggingClassifier(base_estimator=SVC(),
                       n_estimators=10, random_state=0)

clf.fit(x_train,y_train)

ACC = clf.score(x_train,y_train)    
print("Train dataset Accuracy = %"+ str(ACC*100))

ACC = clf.score(x_test,y_test)    
print("Test dataset Accuracy = % "+ str(ACC*100))

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

clf = AdaBoostClassifier(n_estimators=100, random_state=0)

clf.fit(x_train,y_train)

ACC = clf.score(x_train,y_train)    
print("Train dataset Accuracy = %"+ str(ACC*100))

ACC = clf.score(x_test,y_test)    
print("Test dataset Accuracy = % "+ str(ACC*100))

In [ ]:
# Navi Bayes
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(x_train,y_train)

ACC = nb.score(x_train,y_train)    
print("Train dataset Accuracy = %"+ str(ACC*100))

ACC = nb.score(x_test,y_test)    
print("Test dataset Accuracy = % "+ str(ACC*100))

### Conclusion
* **As a result, we learn;**
 *  **Sklearn library**
 *   **Correlation analysis**
 *   **Many Classifier algorithms.**
 *   **Visualization methods**
 *   **I will add new methods to this project in the future.**
 
 
 * ***If you found the content useful, don't forget to upvote and comment. Have a Kaggle day*** 